In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv("qoute_dataset.csv")

In [ ]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [ ]:
df.shape

(3038, 2)

In [ ]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [ ]:
quotes = quotes.str.lower()   # converting all cases to lowercase

In [ ]:
import string
translator = str.maketrans('', '', string.punctuation)  # to replace punctuatin with blank
quotes = quotes.apply(lambda x: x.translate(translator))  # apply the replacement

In [ ]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


## RNN Model

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
vocab_size = 10000

# Creating Tokens
tokinizer = Tokenizer(num_words=vocab_size)
tokinizer.fit_on_texts(quotes)

In [ ]:
# Showing the unique integer index

word_index = tokinizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [ ]:
# Creating Sequence to preserve order

sequence = tokinizer.texts_to_sequences(quotes)

In [ ]:
for i in range(3):
  print("Quotes: ")
  print(quotes[i])
  print("Sequences: ")
  print(sequence[i])

Quotes: 
“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”
Sequences: 
[713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104, 752, 70, 2461]
Quotes: 
“it is our choices harry that show what we truly are far more than our abilities”
Sequences: 
[947, 7, 70, 871, 373, 9, 433, 21, 19, 465, 14, 294, 52, 54, 70, 3676]
Quotes: 
“there are only two ways to live your life one is as though nothing is a miracle the other is as though everything is a miracle”
Sequences: 
[1337, 14, 53, 201, 714, 3, 81, 15, 36, 37, 7, 29, 329, 93, 7, 5, 1157, 1, 101, 7, 29, 329, 126, 7, 5, 3677]


In [ ]:
X = []
y = []

# basically this is using 1 word to predict next word, then 2 words together to predict the 3rd word,
# then 3 words together to predict the 4th word and so on...

for seq in sequence:
  # min range is 1 because we'll need atleast one word (0th index) to start with,
  # and max range is len(seq) to exclude the last value since we need atleast one word in the end to predict
  for i in range(1,len(seq)):
    input_seq = seq[:i]  #
    output_seq = seq[i]  # the next word after i words
    X.append(input_seq)
    y.append(output_seq)

In [ ]:
X[:25]

[[713],
 [713, 62],
 [713, 62, 29],
 [713, 62, 29, 19],
 [713, 62, 29, 19, 16],
 [713, 62, 29, 19, 16, 946],
 [713, 62, 29, 19, 16, 946, 10],
 [713, 62, 29, 19, 16, 946, 10, 7],
 [713, 62, 29, 19, 16, 946, 10, 7, 5],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809],
 [713, 62, 29, 19, 16, 946, 10, 7, 5, 1156, 8, 70, 293, 10, 145, 12, 809, 104],
 [713,
  62,
  29,
  19,
  16,
  946,
  10,
  7,
  5,
  1156,
  8,
  70,
  293,
  10,
  145,
  12,
  809,
  104,
  752],
 [713,
  62,
  29,
  19,
  16,
  946,
  10,
  7,
  5,
  1156,
  8,
  70,
  293,
  10,
  145,
  12,
  809,
  

In [ ]:
y[:20]

[62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [ ]:
len(X)

85271

In [ ]:
len(y)

85271

In [ ]:
max_len = max(len(x) for x in X)  # maximum length of a quote in the dataset
print(max_len)

745


In [ ]:
# Padding

from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X, maxlen=max_len, padding='pre')

X_padded[:21]

array([[  0,   0,   0, ...,   0,   0, 713],
       [  0,   0,   0, ...,   0, 713,  62],
       [  0,   0,   0, ..., 713,  62,  29],
       ...,
       [  0,   0,   0, ..., 809, 104, 752],
       [  0,   0,   0, ..., 104, 752,  70],
       [  0,   0,   0, ...,   0,   0, 947]], dtype=int32)

In [ ]:
X_padded.shape

(85271, 745)

In [ ]:
y = np.array(y) # converting y to array

In [ ]:
# Performing one-hot encoding on 'y' variable
from tensorflow.keras.utils import to_categorical

y_one_hot = to_categorical(y, num_classes=vocab_size)  # vocab_size = 10000

In [ ]:
y.shape

(85271,)

In [ ]:
y_one_hot.shape

(85271, 10000)

In [ ]:
y_one_hot[0]

array([0., 0., 0., ..., 0., 0., 0.])

# Models

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, Dense

In [ ]:
# Creating Embeddings

embedding_dim = 50
rnn_units = 128

## RNN Model

In [ ]:
rnn_model = Sequential()

# adding embedding layer
rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
# adding simple rnn layer / input layer
rnn_model.add(SimpleRNN(units=rnn_units))
# dense layer it is / output layer
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
rnn_model.summary()  # to see the layers

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## LSTM Model

In [ ]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

In [ ]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
lstm_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Training Models

In [ ]:
# RNN Model training code (will take a lot of time)

# epochs = 10
# batch_size = 128

# history_rnn = rnn_model.fit(
#     X_padded, y_one_hot,
#     epochs=epochs,
#     batch_size=batch_size,
#     validation_split = 0.1
# )

In [ ]:
# LSTM Model training code
# Accuracy: 76.22%, validation accuracy: 0.0742
# Loss: 1.0753, validation loss: 12.2574

epochs = 100
batch_size = 128

history_lstm = lstm_model.fit(
    X_padded, y_one_hot,
    epochs=epochs,
    batch_size=batch_size,
    validation_split = 0.1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.1595 - loss: 4.7511 - val_accuracy: 0.1136 - val_loss: 6.6157
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 55ms/step - accuracy: 0.1673 - loss: 4.6235 - val_accuracy: 0.1107 - val_loss: 6.6856
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1777 - loss: 4.4987 - val_accuracy: 0.1106 - val_loss: 6.7492
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1863 - loss: 4.3756 - val_accuracy: 0.1141 - val_loss: 6.8110
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1985 - loss: 4.2550 - val_accuracy: 0.1115 - val_loss: 6.8918
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.2104 - loss: 4.1389 - val_accuracy: 0.1100 - val_loss: 6.9616
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.2262 - loss: 4.0243 - val_accuracy: 0.1112 - val_loss: 7.0383
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 31s 52ms/step - accuracy: 0.2396 - loss: 3

In [ ]:
lstm_model.save("lstm_model.h5")

# Predicting the next word

### Creating word map

Since, we'll get the predictions in numeric form. So, we've to convert those numbers back to text.

In [ ]:
index_to_word = {}
for word, index in word_index.items():
  index_to_word[index] = word

In [ ]:
index_to_word

{1: 'the',
 2: 'you',
 3: 'to',
 4: 'and',
 5: 'a',
 6: 'i',
 7: 'is',
 8: 'of',
 9: 'that',
 10: 'it',
 11: 'in',
 12: 'be',
 13: 'not',
 14: 'are',
 15: 'your',
 16: 'have',
 17: 'for',
 18: 'but',
 19: 'we',
 20: 'if',
 21: 'what',
 22: 'with',
 23: 'all',
 24: 'love',
 25: 'can',
 26: 'my',
 27: 'when',
 28: 'will',
 29: 'as',
 30: 'who',
 31: 'do',
 32: 'or',
 33: 'me',
 34: 'he',
 35: 'they',
 36: 'life',
 37: 'one',
 38: 'was',
 39: 'like',
 40: 'there',
 41: 'people',
 42: 'on',
 43: 'its',
 44: 'at',
 45: 'so',
 46: 'never',
 47: 'no',
 48: 'them',
 49: 'dont',
 50: 'know',
 51: 'just',
 52: 'more',
 53: 'only',
 54: 'than',
 55: 'because',
 56: 'this',
 57: 'want',
 58: 'up',
 59: 'how',
 60: 'his',
 61: 'things',
 62: 'world',
 63: 'by',
 64: 'think',
 65: 'make',
 66: 'about',
 67: 'time',
 68: 'from',
 69: 'always',
 70: 'our',
 71: 'an',
 72: 'out',
 73: 'us',
 74: 'good',
 75: 'said',
 76: 'she',
 77: 'her',
 78: 'way',
 79: 'go',
 80: 'am',
 81: 'live',
 82: 'has',
 83:

In [ ]:
# function to get the predicted word (one word at a time) and returning it in textual form

def predictor(model,tokenizer,text,max_len):
  # 1. tokenizer is used to tokenize the input text we'll receive,
  # and it will tokenize the input similarly to how it did for the dataset
  # 2. text is the input word/words for which we want to predict the next word

  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]  # making the sequence of input text
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')  # padding on the input

  pred = model.predict(seq,verbose = 0)  # predicting the next word
  pred_index = np.argmax(pred)  # getting the index of predicted word
  return index_to_word[pred_index]  # returning the word from index

In [ ]:
seed_text = "the world as we"
next_word = predictor(lstm_model,tokinizer,seed_text,max_len)
print(next_word)

have


In [ ]:
# function to get the predicted word (mulitple words) and returning it in textual form

def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":  # break loop after empty space
      break
    seed_text += " " + next_word
  return seed_text

In [ ]:
seed = "are you a"
generate_text = generate_text(lstm_model,tokinizer,seed,max_len,10)
print(generate_text)

are you a lot of you cared because you are the questions thought


In [ ]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokinizer, f)

In [ ]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)